# Project 2 — Data Fitting and OLS Method
### Part 1: Functions 1 - 4

---


| # | Function | Description |
|---|----------|-------------|
| 1 | `ols_fit` | Estimate coefficients $\hat{\beta}$ and noise variance $\hat{\sigma}^2$ |
| 2 | `hat_matrix` | Compute hat matrix $H$ and verify idempotency |
| 3 | `model_metrics` | Compute RSS, TSS, $R^2$, $\bar{R}^2$, F-statistic |
| 4 | `coef_inference` | Coefficient inference: SE, t-stat, p-value, confidence intervals |

## 0. Imports and Test Data

In [ ]:
import math
import numpy as np
from scipy import stats
from sklearn.linear_model import LinearRegression

import sys
sys.path.insert(0, '.')
from ols_implementation import (
    ols_fit, hat_matrix, model_metrics, coef_inference,
    print_results, print_hat_matrix, print_metrics, print_inference
)

print("Import successful!")

In [ ]:

X1 = [[1.0], [3.0], [4.0], [7.0], [9.0], [12.0]]
y1 = [0.0,   2.0,   5.0,  10.0,  12.0,   16.0]

print("Data is ready.")

---
## Function 1 — `ols_fit`: OLS Estimation

### 1.1 Linear Model

Assume the data follows the model:

$$
\mathbf{y} = \mathbf{X}\boldsymbol{\beta} + \boldsymbol{\varepsilon},
\qquad \boldsymbol{\varepsilon} \sim \mathcal{N}(\mathbf{0},\, \sigma^2 \mathbf{I}_n)
$$

where $\mathbf{X} \in \mathbb{R}^{n \times (p+1)}$ is the **design matrix** (first column all ones for intercept), $\boldsymbol{\beta} \in \mathbb{R}^{p+1}$ is the parameter vector to be estimated.

### 1.2 RSS Loss Function

OLS finds $\hat{\boldsymbol{\beta}}$ that minimizes the **Residual Sum of Squares**:

$$
\text{RSS}(\boldsymbol{\beta})
= \|\mathbf{y} - \mathbf{X}\boldsymbol{\beta}\|_2^2
= \sum_{i=1}^{n}(y_i - \mathbf{x}_i^\top \boldsymbol{\beta})^2
$$

### 1.3 Derivation of Solution (Normal Equations)

Compute the gradient and set to zero:

$$
\nabla_{\boldsymbol{\beta}}\,\text{RSS}
= -2\mathbf{X}^\top(\mathbf{y} - \mathbf{X}\boldsymbol{\beta}) = \mathbf{0}
\;\Longrightarrow\;
\mathbf{X}^\top\mathbf{X}\,\boldsymbol{\beta} = \mathbf{X}^\top\mathbf{y}
$$

When $\mathbf{X}^\top\mathbf{X}$ is invertible (full rank), the unique solution is:

$$
\boxed{\hat{\boldsymbol{\beta}}_{\text{OLS}} = (\mathbf{X}^\top\mathbf{X})^{-1}\mathbf{X}^\top\mathbf{y}}
$$

### 1.4 Noise Variance Estimation

Unbiased estimator of $\sigma^2$:

$$
\boxed{\hat{\sigma}^2 = \frac{\text{RSS}}{n - p - 1}
= \frac{\|\mathbf{y} - \mathbf{X}\hat{\boldsymbol{\beta}}\|^2}{n - p - 1}}
$$

The denominator $n - p - 1$ is the remaining degrees of freedom (having used $p+1$ parameters). Dividing by $n-p-1$ instead of $n$ ensures $\mathbb{E}[\hat{\sigma}^2] = \sigma^2$.

In [ ]:

result1 = ols_fit(X1, y1)
print_results(result1, feature_names=["x"])
print(f"  n = {result1['n']},  p = {result1['p']},  dof = {result1['dof']}")

In [ ]:

X_np = np.array(X1)
y_np = np.array(y1)
X_design = np.hstack([np.ones((len(X_np), 1)), X_np])


beta_np, _, _, _ = np.linalg.lstsq(X_design, y_np, rcond=None)
print("NumPy lstsq  beta:", beta_np)


lr = LinearRegression().fit(X_np, y_np)
print("sklearn      beta:", [lr.intercept_] + list(lr.coef_))

print("Our implementation beta:", result1['beta_hat'])
print("Matches NumPy (beta):", np.allclose(result1['beta_hat'], beta_np, atol=1e-4))

---
## Function 2 — `hat_matrix`: Projection Matrix

### 2.1 Definition

The **hat matrix** (projection matrix) is:

$$
\boxed{H = \mathbf{X}(\mathbf{X}^\top\mathbf{X})^{-1}\mathbf{X}^\top \in \mathbb{R}^{n \times n}}
$$

The name comes from the relation: $\hat{\mathbf{y}} = H\mathbf{y}$ — $H$ "puts a hat" on $\mathbf{y}$.

### 2.2 Key Properties

| Property | Formula | Meaning |
|----------|---------|--------|
| **Idempotent** | $H^2 = H$ | Projecting twice = projecting once |
| **Symmetric** | $H^\top = H$ | Orthogonal projection matrix |
| **Fitted values** | $\hat{\mathbf{y}} = H\mathbf{y}$ | Projects $\mathbf{y}$ onto $\text{col}(\mathbf{X})$ |
| **Residuals** | $\hat{\boldsymbol{\varepsilon}} = (I-H)\mathbf{y}$ | Component orthogonal to $\text{col}(\mathbf{X})$ |
| **Rank** | $\text{rank}(H) = p+1$ | Dimension of the subspace |

### 2.3 Proof of Idempotency

$$
H^2 = \bigl[\mathbf{X}(\mathbf{X}^\top\mathbf{X})^{-1}\mathbf{X}^\top\bigr]
      \bigl[\mathbf{X}(\mathbf{X}^\top\mathbf{X})^{-1}\mathbf{X}^\top\bigr]
= \mathbf{X}\underbrace{(\mathbf{X}^\top\mathbf{X})^{-1}(\mathbf{X}^\top\mathbf{X})}_{=\,I_{p+1}}(\mathbf{X}^\top\mathbf{X})^{-1}\mathbf{X}^\top
= \mathbf{X}(\mathbf{X}^\top\mathbf{X})^{-1}\mathbf{X}^\top = H \quad \blacksquare
$$

In [ ]:

h_result = hat_matrix(X1)
print_hat_matrix(h_result)

In [ ]:

X_d = np.hstack([np.ones((len(X1), 1)), np.array(X1)])
H_np = X_d @ np.linalg.inv(X_d.T @ X_d) @ X_d.T
H_our = np.array(h_result['H'])

print("Max absolute difference (H_numpy vs H_ours):",
      np.max(np.abs(H_np - H_our)))

print("H^2 = H (NumPy check):",
      np.allclose(H_np @ H_np, H_np))

print("H = H^T (symmetry)   :",
      np.allclose(H_np, H_np.T))

print(f"trace(H) = {np.trace(H_np):.4f}  (should be = {len(X1[0])+1})")
print("Matches NumPy (H):", np.allclose(H_our, H_np, atol=1e-4))

---
## Function 3 — `model_metrics`: Model Evaluation

### 3.1 Sum of Squares Decomposition (ANOVA)

$$
\underbrace{\sum_{i=1}^n (y_i - \bar{y})^2}_{\text{TSS}}
= \underbrace{\sum_{i=1}^n (\hat{y}_i - \bar{y})^2}_{\text{MSS (explained)}}
+ \underbrace{\sum_{i=1}^n (y_i - \hat{y}_i)^2}_{\text{RSS (unexplained)}}
$$

### 3.2 Coefficient of Determination $R^2$

$$
\boxed{R^2 = 1 - \frac{\text{RSS}}{\text{TSS}} \in [0, 1]}
$$

**Interpretation:** $R^2$ measures the proportion of variance in $y$ explained by the model.  
- $R^2 = 1$: perfect model fit  
- $R^2 = 0$: model is no better than predicting with $\bar{y}$

**Note:** $R^2$ always increases when adding variables, even if they are insignificant.

### 3.3 Adjusted $R^2$

$$
\boxed{\bar{R}^2 = 1 - \frac{n-1}{n-p-1}(1 - R^2)}
$$

Penalizes adding variables ($p$ increases → $n-p-1$ decreases → $\bar{R}^2$ decreases if the new variable contributes nothing).

### 3.4 F-test (Overall Model Significance)

$$
H_0: \beta_1 = \beta_2 = \cdots = \beta_p = 0
\quad \text{vs} \quad
H_1: \exists\, j,\; \beta_j \neq 0
$$

$$
\boxed{F = \frac{\text{MSS}/p}{\text{RSS}/(n-p-1)}
= \frac{(\text{TSS}-\text{RSS})/p}{\text{RSS}/(n-p-1)}
\sim F_{p,\;n-p-1}}
$$

Large $F$ → reject $H_0$ → model is statistically significant.

In [ ]:
metrics1 = model_metrics(y1, result1['y_hat'], result1['p'])
print_metrics(metrics1)
print(f"  y_mean = {metrics1['y_mean']:.6f}")
print(f"  df_model = {metrics1['df_model']},  df_resid = {metrics1['df_resid']}")

In [ ]:
y_np  = np.array(y1)
yh_np = np.array(result1['y_hat'])
n, p  = result1['n'], result1['p']

rss_np = np.sum((y_np - yh_np)**2)
tss_np = np.sum((y_np - y_np.mean())**2)
r2_np  = 1 - rss_np / tss_np
r2_adj_np = 1 - (n-1)/(n-p-1) * (1 - r2_np)
f_np   = ((tss_np - rss_np)/p) / (rss_np/(n-p-1))

print(f"NumPy  RSS={rss_np:.6f}  R²={r2_np:.6f}  R²_adj={r2_adj_np:.6f}  F={f_np:.6f}")
print(f"Ours   RSS={metrics1['rss']:.6f}  R2={metrics1['r2']:.6f}  R2_adj={metrics1['r2_adj']:.6f}  F={metrics1['f_stat']:.6f}")
print("Matches NumPy:", np.isclose(metrics1['rss'], rss_np, atol=1e-4)
      and np.isclose(metrics1['r2'], r2_np, atol=1e-4)
      and np.isclose(metrics1['f_stat'], f_np, atol=1e-4))

---
## Function 4 — `coef_inference`: Coefficient Inference

### 4.1 Distribution of $\hat{\boldsymbol{\beta}}$

Under Gauss–Markov assumptions (GM1–GM5):

$$
\hat{\boldsymbol{\beta}} \sim \mathcal{N}\!\left(\boldsymbol{\beta},\; \sigma^2(\mathbf{X}^\top\mathbf{X})^{-1}\right)
$$

### 4.2 Standard Error (SE)

$$
\boxed{\text{SE}(\hat{\beta}_j) = \hat{\sigma}\,\sqrt{[(\mathbf{X}^\top\mathbf{X})^{-1}]_{jj}}}
$$

### 4.3 t-test (Student's)

$$
H_0: \beta_j = 0 \quad \text{vs} \quad H_1: \beta_j \neq 0
$$

$$
\boxed{t_j = \frac{\hat{\beta}_j}{\text{SE}(\hat{\beta}_j)} \sim t_{n-p-1}}
\quad \text{(under } H_0\text{)}
$$

- **p-value** (two-tailed): $p_j = 2\,P(T > |t_j|)$ where $T \sim t_{n-p-1}$
- Reject $H_0$ at significance level $\alpha$ when $p_j < \alpha$ (typically $\alpha = 0.05$)

### 4.4 $(1-\alpha)\cdot 100\%$ Confidence Interval

$$
\boxed{\hat{\beta}_j \pm t_{\alpha/2,\;n-p-1} \cdot \text{SE}(\hat{\beta}_j)}
$$

For a 95% confidence level, $\alpha = 0.05$ and $t_{0.025, n-p-1}$ is the quantile of the $t$-distribution.

### 4.5 Process Diagram

```
X, y, β̂, σ̂²
     │
     ├─→ (XᵀX)⁻¹  ──→  SE(β̂ⱼ) = σ̂√[(XᵀX)⁻¹]ⱼⱼ
     │                              │
     │              ┌───────────────┴──────────────┐
     │              ↓                              ↓
     │         tⱼ = β̂ⱼ/SE(β̂ⱼ)          CI: β̂ⱼ ± t_crit·SE
     │              │
     │              ↓
     │         p-value = 2·P(T > |tⱼ|)
```

In [ ]:

inf1 = coef_inference(X1, y1, result1['beta_hat'], result1['sigma2'])
print_inference(inf1, feature_names=['x'], beta_hat=result1['beta_hat'])
print()
print(f"  t_critical (α/2=0.025, dof={inf1['dof']}): {inf1['t_crit']:.4f}")
print()
print("  95% Confidence Intervals:")
names = ['intercept', 'x']
for j, name in enumerate(names):
    print(f"    {name:>10s}: [{inf1['ci_lower'][j]:+.4f},  {inf1['ci_upper'][j]:+.4f}]")

In [ ]:

X_d   = np.hstack([np.ones((len(X1),1)), np.array(X1)])
y_np  = np.array(y1)
n, k  = X_d.shape
dof   = n - k

beta_np = np.linalg.inv(X_d.T @ X_d) @ X_d.T @ y_np
yh_np   = X_d @ beta_np
rss_np  = np.sum((y_np - yh_np)**2)
s2_np   = rss_np / dof

se_np  = np.sqrt(s2_np * np.diag(np.linalg.inv(X_d.T @ X_d)))
t_np   = beta_np / se_np
pv_np  = 2 * (1 - stats.t.cdf(np.abs(t_np), dof))

print("        coef      SE      t-stat   p-value")
print("         (scipy reference)")
for j, name in enumerate(['intercept','x']):
    print(f"  {name:>10s}  {se_np[j]:.4f}  {t_np[j]:7.4f}  {pv_np[j]:.4f}")

print()
print("        coef      SE      t-stat   p-value")
print("         (our implementation)")
for j, name in enumerate(['intercept','x']):
    print(f"  {name:>10s}  {inf1['se'][j]:.4f}  {inf1['t_stats'][j]:7.4f}  {inf1['p_values'][j]:.4f}")

print()
print("Matches SciPy (se, p-value):",
      np.allclose(inf1['se'], se_np, atol=1e-4),
      np.allclose(inf1['p_values'], pv_np, atol=1e-4))

---
## Summary

| Function | Main Formula | Verification Result |
|----------|--------------|---------------------|
| `ols_fit` | $\hat{\boldsymbol{\beta}} = (\mathbf{X}^\top\mathbf{X})^{-1}\mathbf{X}^\top\mathbf{y}$, $\hat{\sigma}^2 = \text{RSS}/(n-p-1)$ | Matches NumPy/sklearn |
| `hat_matrix` | $H = \mathbf{X}(\mathbf{X}^\top\mathbf{X})^{-1}\mathbf{X}^\top$, $H^2=H$ | Idempotent, symmetric |
| `model_metrics` | $R^2 = 1 - \text{RSS/TSS}$, $\bar{R}^2$, $F$-stat | Matches NumPy |
| `coef_inference` | $t_j = \hat{\beta}_j/\text{SE}_j$, CI $= \hat{\beta}_j \pm t_{\alpha/2}\cdot\text{SE}_j$ | Matches scipy.stats |